### Import Library

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [7]:
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

### Load Dataset

In [8]:
# Read dataset csv
df = pd.read_csv("transactions.csv")
df.head()

,transaction_id,user_id,account_age_days,total_transactions_user,avg_amount_user,amount,country,bin_country,channel,merchant_category,promo_used,avs_match,cvv_result,three_ds_flag,transaction_time,shipping_distance_km,is_fraud
0,1,1,141,47,147.93,84.75,FR,FR,web,travel,0,1,1,1,2024-01-06T04:09:39Z,370.95,0
1,2,1,141,47,147.93,107.90,FR,FR,web,travel,0,0,0,0,2024-01-09T20:13:47Z,149.62,0
2,3,1,141,47,147.93,92.36,FR,FR,app,travel,1,1,1,1,2024-01-12T06:20:11Z,164.08,0
3,4,1,141,47,147.93,112.47,FR,FR,web,fashion,0,1,1,1,2024-01-15T17:00:04Z,397.40,0
4,5,1,141,47,147.93,132.91,FR,US,web,electronics,0,1,1,1,2024-01-17T01:27:31Z,935.28,0


### Problem & Dataset Understanding

Membangun model untuk mengidentifikasi transaksi yang berpotensi fraud berdasarkan karakteristik transaksi, user, payment verification, channel dan lokasi.

Target: [is_fraud]

In [12]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

Shape: (9999, 17)

Columns:
['transaction_id', 'user_id', 'account_age_days', 'total_transactions_user', 'avg_amount_user', 'amount', 'country', 'bin_country', 'channel', 'merchant_category', 'promo_used', 'avs_match', 'cvv_result', 'three_ds_flag', 'transaction_time', 'shipping_distance_km', 'is_fraud']

Data Types:
transaction_id               int64
user_id                      int64
account_age_days             int64
total_transactions_user      int64
avg_amount_user            float64
amount                     float64
country                     object
bin_country                 object
channel                     object
merchant_category           object
promo_used                   int64
avs_match                    int64
cvv_result                   int64
three_ds_flag                int64
transaction_time            object
shipping_distance_km       float64
is_fraud                     int64
dtype: object


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   transaction_id           9999 non-null   int64  
 1   user_id                  9999 non-null   int64  
 2   account_age_days         9999 non-null   int64  
 3   total_transactions_user  9999 non-null   int64  
 4   avg_amount_user          9999 non-null   float64
 5   amount                   9999 non-null   float64
 6   country                  9999 non-null   object 
 7   bin_country              9999 non-null   object 
 8   channel                  9999 non-null   object 
 9   merchant_category        9999 non-null   object 
 10  promo_used               9999 non-null   int64  
 11  avs_match                9999 non-null   int64  
 12  cvv_result               9999 non-null   int64  
 13  three_ds_flag            9999 non-null   int64  
 14  transaction_time        

In [15]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
transaction_id,9999.0,5000.000000,2886.607005,1.00,2500.500,5000.00,7499.500,9999.00
user_id,9999.0,100.491849,58.252421,1.00,49.000,100.00,151.000,201.00
account_age_days,9999.0,1022.828883,519.212191,10.00,566.000,1085.00,1449.000,1888.00
total_transactions_user,9999.0,50.516552,5.825915,40.00,46.000,51.00,56.000,60.00
avg_amount_user,9999.0,136.492350,169.633322,6.56,44.060,90.48,147.930,1366.72
amount,9999.0,162.679380,265.747270,1.16,41.840,87.68,174.635,3883.46
promo_used,9999.0,0.149015,0.356121,0.00,0.000,0.00,0.000,1.00
avs_match,9999.0,0.842184,0.364586,0.00,1.000,1.00,1.000,1.00
cvv_result,9999.0,0.879688,0.325342,0.00,1.000,1.00,1.000,1.00
three_ds_flag,9999.0,0.782778,0.412375,0.00,1.000,1.00,1.000,1.00


### Data Quality

In [20]:
missing = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": df.isnull().mean() * 100
})

missing = missing.sort_values(
    "missing_count",
    ascending=False
)

missing

,missing_count,missing_percentage
transaction_id,0,0.0
user_id,0,0.0
account_age_days,0,0.0
total_transactions_user,0,0.0
avg_amount_user,0,0.0
amount,0,0.0
country,0,0.0
bin_country,0,0.0
channel,0,0.0
merchant_category,0,0.0


In [23]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate transaction_id:", df["transaction_id"].duplicated().sum())

Duplicate rows: 0
Duplicate transaction_id: 0


In [24]:
df.nunique().sort_values()

cvv_result                    2
avs_match                     2
promo_used                    2
three_ds_flag                 2
channel                       2
is_fraud                      2
merchant_category             5
country                      10
bin_country                  10
total_transactions_user      21
account_age_days            193
avg_amount_user             200
user_id                     201
amount                     8263
shipping_distance_km       9161
transaction_time           9998
transaction_id             9999
dtype: int64

In [26]:
df["transaction_time"] = pd.to_datetime(
    df["transaction_time"],
    errors="coerce"
)
df.dtypes

transaction_id                           int64
user_id                                  int64
account_age_days                         int64
total_transactions_user                  int64
avg_amount_user                        float64
amount                                 float64
country                                 object
bin_country                             object
channel                                 object
merchant_category                       object
promo_used                               int64
avs_match                                int64
cvv_result                               int64
three_ds_flag                            int64
transaction_time           datetime64[ns, UTC]
shipping_distance_km                   float64
is_fraud                                 int64
dtype: object

In [28]:
df["transaction_time"].isna().sum()

0

### Univariate Analysis

In [31]:
numeric_cols = [
    "total_transactions_user",
    "account_age_days",
    "avg_amount_user",
    "amount",
    "shipping_distance_km"
]

#### 1. Summary Statistics

In [32]:
df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
total_transactions_user,9999.0,50.516552,5.825915,40.00,46.000,51.00,56.000,60.00
account_age_days,9999.0,1022.828883,519.212191,10.00,566.000,1085.00,1449.000,1888.00
avg_amount_user,9999.0,136.492350,169.633322,6.56,44.060,90.48,147.930,1366.72
amount,9999.0,162.679380,265.747270,1.16,41.840,87.68,174.635,3883.46
shipping_distance_km,9999.0,351.895818,416.013336,0.07,135.515,273.47,407.645,3715.80


In [34]:
summary = df[numeric_cols].describe().T
summary["median"] = df[numeric_cols].median()
summary

,count,mean,std,min,25%,50%,75%,max,median
total_transactions_user,9999.0,50.516552,5.825915,40.00,46.000,51.00,56.000,60.00,51.00
account_age_days,9999.0,1022.828883,519.212191,10.00,566.000,1085.00,1449.000,1888.00,1085.00
avg_amount_user,9999.0,136.492350,169.633322,6.56,44.060,90.48,147.930,1366.72,90.48
amount,9999.0,162.679380,265.747270,1.16,41.840,87.68,174.635,3883.46,87.68
shipping_distance_km,9999.0,351.895818,416.013336,0.07,135.515,273.47,407.645,3715.80,273.47


#### 2. Mean vs Median

In [35]:
df[numeric_cols].agg(["mean", "median", "std", "min", "max"]).T

,mean,median,std,min,max
total_transactions_user,50.516552,51.00,5.825915,40.00,60.00
account_age_days,1022.828883,1085.00,519.212191,10.00,1888.00
avg_amount_user,136.492350,90.48,169.633322,6.56,1366.72
amount,162.679380,87.68,265.747270,1.16,3883.46
shipping_distance_km,351.895818,273.47,416.013336,0.07,3715.80


#### 3. Skewness

In [36]:
df[numeric_cols].skew().sort_values(ascending=False)

amount                     5.597721
avg_amount_user            3.926146
shipping_distance_km       3.511942
total_transactions_user   -0.110570
account_age_days          -0.120631
dtype: float64

#### IQR

In [37]:
Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)

IQR = Q3 = Q1

iqr_summary = pd.DataFrame({
    "Q1": Q1,
    "Q3": Q3,
    "IQR": IQR
})

iqr_summary

,Q1,Q3,IQR
total_transactions_user,46.000,46.000,46.000
account_age_days,566.000,566.000,566.000
avg_amount_user,44.060,44.060,44.060
amount,41.840,41.840,41.840
shipping_distance_km,135.515,135.515,135.515
